# Lab 02: Linear and Softmax Regression from Scratch & PyTorch `nn.Module`

Welcome to Laboratory 02! In this lab, we build two foundational models of supervised machine learning:
1. **Univariate & Multivariate Linear Regression** (Regression for continuous targets).
2. **Softmax (Multinomial Logistic) Regression** (Classification for multi-class categorical targets on MNIST digits).

### Learning Objectives
* Implement the analytical gradient descent update rule from first principles using pure PyTorch tensors.
* Re-implement models using PyTorch's object-oriented `torch.nn.Module` architecture.
* Understand loss functions: **Mean Squared Error (MSE)** vs. **Cross-Entropy Loss**.
* Train, evaluate, and visualize the spatial weight templates learned by Softmax Regression.


## 1. Technical Preliminaries & Environment Configuration


In [ ]:
# Import core libraries for deep learning, datasets, transformations, and plotting
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set deterministic seed for reproducible results
torch.manual_seed(42)
np.random.seed(42)

# Select computation device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Linear Regression from First Principles (Pure Tensor Operations)

### Mathematical Formulation & Architecture
For a linear relationship with ground truth parameters $w^* = 2.5$ and $b^* = 1.0$:
* **Model Prediction**: $\hat{y}^{(i)} = w x^{(i)} + b$
* **Objective Function (Mean Squared Error Loss)**:
  $$\mathcal{L}(w, b) = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}^{(i)} - y^{(i)})^2 = \frac{1}{N} \sum_{i=1}^{N} (w x^{(i)} + b - y^{(i)})^2$$
* **Parameter Update Rule (Stochastic Gradient Descent)**:
  $$w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w}, \quad b \leftarrow b - \eta \frac{\partial \mathcal{L}}{\partial b}$$
  where $\eta$ is the learning rate.


In [ ]:
# Step 1: Generate synthetic 1D dataset: y = 2.5 * x + 1.0 + Gaussian Noise
num_samples = 100
X_true = torch.linspace(-3, 3, num_samples).view(-1, 1)                      # Shape: (100, 1)
noise = 0.5 * torch.randn(num_samples, 1)                                   # Gaussian noise N(0, 0.25)
y_true = 2.5 * X_true + 1.0 + noise                                         # Ground truth targets (100, 1)

# Step 2: Initialize learnable parameters randomly with gradient tracking enabled
w_scratch = torch.randn(1, 1, requires_grad=True)                           # Trainable weight
b_scratch = torch.zeros(1, requires_grad=True)                              # Trainable bias

learning_rate = 0.05                                                        # Step size for SGD
num_epochs = 100                                                            # Number of optimization epochs

# Step 3: Manual Training Loop using Autograd & Tensor Operations
for epoch in range(num_epochs):
    # Forward Pass: Compute model linear prediction y_pred = X * w + b
    y_pred = X_true @ w_scratch + b_scratch
    
    # Compute Mean Squared Error (MSE) loss
    loss = torch.mean((y_pred - y_true) ** 2)
    
    # Backward Pass: Calculate analytical gradients dL/dw and dL/db
    loss.backward()
    
    # Parameter Update Step without tracking operations in computational graph
    with torch.no_grad():
        w_scratch -= learning_rate * w_scratch.grad
        b_scratch -= learning_rate * b_scratch.grad
        
        # Zero gradients after update to avoid accumulation in next iteration
        w_scratch.grad.zero_()
        b_scratch.grad.zero_()
        
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1:03d}/{num_epochs}] | MSE Loss: {loss.item():.4f} | w: {w_scratch.item():.3f} | b: {b_scratch.item():.3f}')

print(f'Final Learned Parameters -> w: {w_scratch.item():.4f} (True: 2.5), b: {b_scratch.item():.4f} (True: 1.0)')


## 3. Modular Linear Regression with PyTorch `nn.Module`

### Architecture Overview: `nn.Linear` & `torch.optim`
Here, we encapsulate the linear model using PyTorch's standardized `nn.Linear` module, which automatically manages internal weight matrix $\mathbf{W}$ and bias vector $\mathbf{b}$, paired with `nn.MSELoss` and `optim.SGD`.


In [ ]:
# Instantiate a standard linear regression model: 1 input feature -> 1 output feature
model_linear = nn.Linear(in_features=1, out_features=1)

# Define loss criterion (Mean Squared Error) and optimization algorithm (SGD)
criterion = nn.MSELoss()
optimizer = optim.SGD(model_linear.parameters(), lr=0.05)

# PyTorch Standard Training Loop
for epoch in range(100):
    # 1. Forward Pass: compute predictions through the nn.Linear layer
    predictions = model_linear(X_true)
    
    # 2. Compute Loss
    loss = criterion(predictions, y_true)
    
    # 3. Backward Pass: Clear old gradients and compute new gradients
    optimizer.zero_grad()
    loss.backward()
    
    # 4. Optimizer Step: Update internal model parameters (W and b)
    optimizer.step()

# Retrieve learned weights from nn.Linear parameters
learned_w = model_linear.weight.item()
learned_b = model_linear.bias.item()
print(f'PyTorch nn.Linear Learned Parameters -> w: {learned_w:.4f}, b: {learned_b:.4f}')


## 4. Softmax Regression (Multinomial Logistic Classification) on MNIST

### Architecture & Mathematical Formulation: Softmax Classifier
For an input image flattened into vector $\mathbf{x} \in \mathbb{R}^{784}$ and $C=10$ output classes:
1. **Linear Logits**: $\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$, where $\mathbf{W} \in \mathbb{R}^{10 \times 784}$ and $\mathbf{b} \in \mathbb{R}^{10}$.
2. **Softmax Normalization**:
   $$P(y = c \mid \mathbf{x}) = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}$$
3. **Cross-Entropy Loss Function**:
   $$\mathcal{L}_{CE} = -\sum_{c=1}^{C} y_c \log(P(y = c \mid \mathbf{x})) = -\log(P(y = y_{true} \mid \mathbf{x}))$$
In PyTorch, `nn.CrossEntropyLoss()` combines `nn.LogSoftmax()` and `nn.NLLLoss()` in a single numerically stable function.


### Softmax Regression Model Class Definition: `SoftmaxRegression`
The following cell defines the custom neural network class `SoftmaxRegression` inheriting from `nn.Module`.
* **Input**: Batched 2D images of shape `(Batch_Size, 1, 28, 28)`.
* **Transform**: Flattens input tensor to `(Batch_Size, 784)`.
* **Linear Layer**: Maps 784 input dimensions to 10 class logits `(Batch_Size, 10)`.


In [ ]:
# Define the Softmax Regression (Single Layer Neural Classifier) Architecture
class SoftmaxRegression(nn.Module):
    """Single-layer neural network for multi-class classification on flattened images."""
    def __init__(self, input_dim: int = 784, num_classes: int = 10):
        super(SoftmaxRegression, self).__init__()
        # Linear projection layer mapping flattened image pixels to class logits
        self.linear = nn.Linear(input_dim, num_classes)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Flatten input from shape (B, 1, 28, 28) -> (B, 784)
        x_flat = x.view(x.size(0), -1)
        # Compute unnormalized logits: shape (B, 10)
        logits = self.linear(x_flat)
        return logits

# Data Preprocessing: Convert PIL images to PyTorch Tensors and normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST standard mean and standard deviation
])

# Download and load MNIST training and validation datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Instantiate DataLoaders for mini-batch stochastic gradient descent
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=256, shuffle=False)

# Instantiate model, loss criterion, and optimizer
clf_model = SoftmaxRegression(input_dim=784, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf_model.parameters(), lr=0.01, momentum=0.9)

print('Model Architecture:\n', clf_model)


### Softmax Regression Training & Evaluation Routine
The following cell executes mini-batch training across multiple epochs and evaluates classification accuracy on unseen test data.


In [ ]:
# Train the Softmax Classifier on MNIST
num_epochs = 3
clf_model.train() # Set model to training mode

for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Move batch data to active compute device (GPU/CPU)
        images, labels = images.to(device), labels.to(device)
        
        # 1. Forward pass: compute class logits
        outputs = clf_model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward pass: compute gradients
        optimizer.zero_grad()
        loss.backward()
        
        # 3. Update parameters
        optimizer.step()
        
        # Compute batch training statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, dim=1) # Get predicted class indices
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0
    print(f'Epoch [{epoch+1}/{num_epochs}] -> Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%')


## 5. Model Evaluation and Learned Weight Template Visualizations

### Pedagogical Insight: Visualizing Classifier Weights as Spatial Filter Templates
Each row in the weight matrix $\mathbf{W} \in \mathbb{R}^{10 \times 784}$ corresponds to a specific digit class (0 through 9). By reshaping each 784-dimensional weight row back into a $28 \times 28$ image, we can directly inspect what spatial patterns each class neuron responds to!


In [ ]:
# Evaluate final test accuracy on held-out test dataset
clf_model.eval() # Switch to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # Disable gradient computation during evaluation for memory efficiency
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = clf_model(images)
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_acc = (correct / total) * 100.0
print(f'Final Test Accuracy on 10,000 MNIST Images: {test_acc:.2f}%')

# Extract learned weight tensor: shape (10, 784)
weights = clf_model.linear.weight.detach().cpu().numpy()

# Visualize the 10 learned spatial weight templates
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.flat):
    # Reshape 784-element row into 28x28 2D spatial grid
    weight_img = weights[digit].reshape(28, 28)
    im = ax.imshow(weight_img, cmap='seismic', vmin=-np.max(np.abs(weights)), vmax=np.max(np.abs(weights)))
    ax.set_title(f'Digit {digit} Weights')
    ax.axis('off')

plt.suptitle('Learned Spatial Weight Filters (Positive=Red, Negative=Blue)', fontsize=13)
plt.tight_layout()
plt.show()


## 6. Summary & Key Takeaways
1. **Linear Regression** optimizes continuous outputs using Mean Squared Error (MSE).
2. **Softmax Regression** generalizes logistic regression to multi-class classification by outputting probability distributions normalized via Softmax.
3. **`nn.CrossEntropyLoss`** internally combines Log-Softmax and Negative Log-Likelihood Loss for superior numerical stability.
4. **Weight Visualizations** reveal that linear models learn digit templates with positive weights in the digit stroke center and negative weights in surrounding regions.
